# 03 — Financial Sentiment Analysis (Loughran-McDonald)

**Goal:** load the Loughran-McDonald dictionary, score each company's MD&A and Risk Factors text
for Negative/Positive/Uncertainty/Litigious/Constraining language, and save the results.

**Before running this notebook:** download `LoughranMcDonald_MasterDictionary_1993-2025.csv` from
https://sraf.nd.edu/loughranmcdonald-master-dictionary/ and place it at
`dictionaries/loughran_mcdonald.csv`.

## Setup

In [1]:
import os

PROJECT_ROOT = r"C:\Users\Devanshi\financial-nlp-analyzer"
os.chdir(PROJECT_ROOT)
print("Working directory set to:", os.getcwd())

Working directory set to: C:\Users\Devanshi\financial-nlp-analyzer


In [2]:
import os
import re
import pandas as pd

EXTRACTED_DIR = "data/extracted_text"
PROCESSED_DIR = "data/processed"
DICT_PATH = "dictionaries/Loughran-McDonald_MasterDictionary_1993-2025.csv"

COMPANIES = ["jpmorgan", "caterpillar", "accenture", "pg", "infosys"]


## Load extracted text from disk

In [3]:
def load_section(company, section):
    path = os.path.join(EXTRACTED_DIR, f"{company}_{section}.txt")
    if not os.path.exists(path):
        print(f"WARNING: {path} not found — skipping")
        return None
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

texts = {}
for company in COMPANIES:
    texts[company] = {
        "mdna": load_section(company, "mdna"),
        "risk": load_section(company, "risk"),
    }


## Load the Loughran-McDonald dictionary

In [4]:
def load_lm_dictionary(filepath=DICT_PATH):
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"{filepath} not found. Download it from "
            "https://sraf.nd.edu/loughranmcdonald-master-dictionary/ and place it there."
        )
    lm = pd.read_csv(filepath)
    lm.columns = [c.strip() for c in lm.columns]
    lm["Word"] = lm["Word"].str.upper()

    categories = {}
    for cat in ["Negative", "Positive", "Uncertainty", "Litigious", "Constraining"]:
        categories[cat] = set(lm.loc[lm[cat] > 0, "Word"])
    return categories

lm_categories = load_lm_dictionary()
for cat, words in lm_categories.items():
    print(f"{cat}: {len(words):,} words loaded")


Negative: 2,345 words loaded
Positive: 347 words loaded
Uncertainty: 297 words loaded
Litigious: 903 words loaded
Constraining: 184 words loaded


## Sentiment scoring function

In [5]:
def score_sentiment(text, lm_categories):
    """
    Returns raw counts AND percentage-of-total-words for each LM category.
    Always compare companies using the _pct values, not raw counts —
    raw counts are meaningless across documents of different lengths.
    """
    if text is None:
        return None

    words = re.findall(r'\b[A-Za-z]+\b', text.upper())
    total_words = len(words)
    scores = {"total_words": total_words}

    for cat, wordset in lm_categories.items():
        count = sum(1 for w in words if w in wordset)
        scores[cat] = count
        scores[f"{cat}_pct"] = round(100 * count / total_words, 3) if total_words else 0

    return scores


## Score every company's MD&A and Risk Factors text

In [6]:
sentiment_records = []

for company in COMPANIES:
    mdna_scores = score_sentiment(texts[company]["mdna"], lm_categories)
    risk_scores = score_sentiment(texts[company]["risk"], lm_categories)

    if mdna_scores is None:
        print(f"Skipping {company} — no MD&A text found")
        continue

    row = {"company": company}
    row.update({f"mdna_{k}": v for k, v in mdna_scores.items()})
    if risk_scores:
        row.update({f"risk_{k}": v for k, v in risk_scores.items()})

    sentiment_records.append(row)

sentiment_df = pd.DataFrame(sentiment_records)
sentiment_df


,company,mdna_total_words,mdna_Negative,mdna_Negative_pct,mdna_Positive,mdna_Positive_pct,mdna_Uncertainty,mdna_Uncertainty_pct,mdna_Litigious,mdna_Litigious_pct,...,risk_Negative,risk_Negative_pct,risk_Positive,risk_Positive_pct,risk_Uncertainty,risk_Uncertainty_pct,risk_Litigious,risk_Litigious_pct,risk_Constraining,risk_Constraining_pct
0,jpmorgan,91,0,0.000,0,0.000,5,5.495,1,1.099,...,1137,5.879,89,0.460,578,2.989,385,1.991,229,1.184
1,caterpillar,29091,630,2.166,187,0.643,577,1.983,270,0.928,...,389,3.517,69,0.624,345,3.119,176,1.591,115,1.040
2,accenture,6483,61,0.941,47,0.725,91,1.404,52,0.802,...,614,4.414,145,1.042,513,3.688,304,2.185,123,0.884
3,pg,13307,249,1.871,138,1.037,239,1.796,50,0.376,...,268,4.787,58,1.036,173,3.090,69,1.232,48,0.857
4,infosys,672,5,0.744,2,0.298,10,1.488,2,0.298,...,8,0.225,60,1.684,29,0.814,4,0.112,3,0.084


## Sanity check — always eyeball this before trusting the numbers

Print a few of the actual words that got matched, for one company, so you can confirm the
categorization makes sense and isn't just matching noise.

In [7]:
sample_company = COMPANIES[0]
sample_text = texts[sample_company]["mdna"]

if sample_text:
    words_upper = set(re.findall(r'\b[A-Za-z]+\b', sample_text.upper()))
    for cat in ["Negative", "Uncertainty", "Litigious"]:
        matched = words_upper & lm_categories[cat]
        print(f"{sample_company} — {cat} words found (sample): {list(matched)[:15]}")


jpmorgan — Negative words found (sample): []
jpmorgan — Uncertainty words found (sample): ['RISK', 'APPEAR', 'APPEARS']
jpmorgan — Litigious words found (sample): ['THERETO']


## Save results

In [8]:
out_path = os.path.join(PROCESSED_DIR, "sentiment_scores.csv")
sentiment_df.to_csv(out_path, index=False)
print(f"Saved {out_path}")


Saved data/processed\sentiment_scores.csv
